<a href="https://colab.research.google.com/github/MarlzRana/machine-learning/blob/main/mod_unet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UNet (modified)

This notebook will implement the UNet CNN as described in the [original paper](https://arxiv.org/pdf/1505.04597.pdf) **with modifications**

Modifications:
- Ensure the output mask resolution is the same as the input resolution ($H_{input} = H_{mask}$ and $W_{input} = W_{mask}$)
  - Adding a $padding=1$ to Conv2dBatchNormReLU
  - Changing the MaxPool layer kernel size to 3 and adding adding a $padding=1$
  - Changing the UpConvolvultion:
    - kernel_size: 2->3
    - padding: 0->1
  - Making the copy and crop function perform the resize to upscale whatever is the smallest (the original assumes the previous feature map should always be sized up to the input dimensions)
- Add a sigmoid activation function to the tail of the model

## Imports

External Imports

In [ ]:
import os

import torch

from torch.utils.data import Dataset
from torch import nn
from torch.utils.data import DataLoader
from torchvision.io import read_image
from torchvision import transforms

import matplotlib.pyplot as plt

import cv2

import numpy as np

import albumentations as A

from typing import List, Dict

Internal Imports

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/dataset.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/global_modules.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/unet.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/loss.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/resnet.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/segformer_encoder.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/epochs.ipynb"

## Constants

In [ ]:
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

## ModUNetBlock

In [ ]:
class ModUNetBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super().__init__()
    self.conv_1 = Conv2dBatchNormReLU(
        in_channels=in_channels,
        out_channels=out_channels,
        kernel_size=3,
        padding=1,
    )

    self.conv_2 = Conv2dBatchNormReLU(
        in_channels=out_channels,
        out_channels=out_channels,
        kernel_size=3,
        padding=1,
    )

  def forward(self, x):
    out_conv_1 = self.conv_1(x)
    out_conv_2 = self.conv_2(out_conv_1)

    return out_conv_2

## ModUNetEncoder

In [ ]:
class ModUNetEncoder(nn.Module):
  def __init__(self, channels:List[int], max_pool_conf: Dict):
    super().__init__()

    assert len(channels) == 6, "c:ModUNetEncoder p:channels needs to be of length 6"
    assert "kernel_size" in max_pool_conf, "c:ModUNetEncoder p:max_pool_conf need to have a kernel_size key"
    assert "stride" in max_pool_conf, "c:ModUNetEncoder p:max_pool_conf need to have a stride key"
    assert "padding" in max_pool_conf, "c:ModUNetEncoder p:max_pool_conf need to have a padding key"

    self.channels = channels
    self.max_pool_conf = max_pool_conf

    self.block_1 = ModUNetBlock(
        in_channels=self.channels[0],
        out_channels=self.channels[1],
    )

    self.max_pool_1 = nn.MaxPool2d(
        kernel_size=self.max_pool_conf["kernel_size"],
        stride=self.max_pool_conf["stride"],
        padding=self.max_pool_conf["padding"]
    )

    self.block_2 = ModUNetBlock(
        in_channels=self.channels[1],
        out_channels=self.channels[2],
    )
    self.max_pool_2 = nn.MaxPool2d(
        kernel_size=self.max_pool_conf["kernel_size"],
        stride=self.max_pool_conf["stride"],
        padding=self.max_pool_conf["padding"]
    )

    self.block_3 = ModUNetBlock(
        in_channels=self.channels[2],
        out_channels=self.channels[3]
    )
    self.max_pool_3 = nn.MaxPool2d(
        kernel_size=self.max_pool_conf["kernel_size"],
        stride=self.max_pool_conf["stride"],
        padding=self.max_pool_conf["padding"]
    )

    self.block_4 = ModUNetBlock(
        in_channels=self.channels[3],
        out_channels=self.channels[4]
    )


  def forward(self, x):
    out_block_1 = self.block_1(x)
    out_max_pool_1 = self.max_pool_1(out_block_1)

    out_block_2 = self.block_2(out_max_pool_1)
    out_max_pool_2 = self.max_pool_2(out_block_2)

    out_block_3 = self.block_3(out_max_pool_2)
    out_max_pool_3 = self.max_pool_3(out_block_3)

    out_block_4 = self.block_4(out_max_pool_3)

    return (out_block_4, out_block_3, out_block_2, out_block_1)

## ModUNetCenter

In [ ]:
class ModUNetCenter(nn.Module):
  def __init__(self, channels:List[int], max_pool_conf:Dict, up_conv_conf: dict):
    super().__init__()

    assert len(channels) == 6, "c:ModUNetCenter p:channels needs to be of length 6"

    assert "kernel_size" in max_pool_conf, "c:ModUNetCenter p:max_pool_conf need to have a kernel_size key"
    assert "stride" in max_pool_conf, "c:ModUNetCenter p:max_pool_conf need to have a stride key"
    assert "padding" in max_pool_conf, "c:ModUNetCenter p:max_pool_conf need to have a padding key"

    assert "kernel_size" in up_conv_conf, "c:ModUNetCenter p:up_conv_conf need to have a kernel_size key"
    assert "stride" in up_conv_conf, "c:ModUNetCenter p:up_conv_conf need to have a stride key"
    assert "padding" in up_conv_conf, "c:ModUNetCenter p:up_conv_conf need to have a padding key"

    self.channels = channels
    self.max_pool_conf = max_pool_conf
    self.up_conv_conf = up_conv_conf

    self.max_pool_1 = nn.MaxPool2d(
        kernel_size=self.max_pool_conf["kernel_size"],
        stride=self.max_pool_conf["stride"],
        padding=self.max_pool_conf["padding"]
    )

    self.block_1 = ModUNetBlock(
        in_channels=self.channels[4],
        out_channels=self.channels[5],
    )

    self.up_conv_1 = nn.ConvTranspose2d(
        in_channels=self.channels[5],
        out_channels=self.channels[4],
        kernel_size=self.up_conv_conf["kernel_size"],
        stride=self.up_conv_conf["stride"],
        padding=self.up_conv_conf["padding"]
    )

  def forward(self, x):
    out_max_pool_1 = self.max_pool_1(x)
    out_block_1 = self.block_1(out_max_pool_1)

    return self.up_conv_1(out_block_1)

## ModUNetDecoder

In [ ]:
class ModUNetDecoder(nn.Module):
  def __init__(
      self,
      encoder_name:str,
      channels:List[int],
      up_conv_conf:Dict,
      channel_mismatch_strategy:str=None
    ):
    super().__init__()

    assert channel_mismatch_strategy == None or channel_mismatch_strategy == "crop" or channel_mismatch_strategy == "upconv", "c:ModUnetDecoder channel_mismatch_strategy==None or channel_mismatch_strategy==\"crop\" or channel_mismatch_strategy==\"upconv\""

    assert len(channels) == 6, "c:ModUNetEncoder p:channels needs to be of length 6"

    assert "kernel_size" in up_conv_conf, "c:ModUNetDecoder p:up_conv_conf need to have a kernel_size key"
    assert "stride" in up_conv_conf, "c:ModUNetDecoder p:up_conv_conf need to have a stride key"
    assert "padding" in up_conv_conf, "c:ModUNetDecoder p:up_conv_conf need to have a padding key"

    if encoder_name == "segformer":
      assert channel_mismatch_strategy != None, "c:ModUnetDecoder channel_mismatch_strategy!=None when encoder_name=segformer"

    self.encoder_name = encoder_name
    self.channel_mismatch_strategy = channel_mismatch_strategy
    self.channels = channels
    self.up_conv_conf = up_conv_conf

    self.block_1 = ModUNetBlock(
        in_channels=self.channels[5],
        out_channels=self.channels[4]
    )
    self.up_conv_1 = nn.ConvTranspose2d(
        in_channels=self.channels[4],
        out_channels=self.channels[3],
        kernel_size=self.up_conv_conf["kernel_size"],
        stride=self.up_conv_conf["stride"],
        padding=self.up_conv_conf["padding"],
    )

    if self.encoder_name == "segformer" and self.channel_mismatch_strategy == "upconv":
      self.up_conv_320_256_channel = nn.ConvTranspose2d(
          in_channels=320,
          out_channels=256,
          kernel_size=3,
          stride=1,
          padding=1
      )

    self.block_2 = ModUNetBlock(
        in_channels=self.channels[4],
        out_channels=self.channels[3],
    )
    self.up_conv_2 = nn.ConvTranspose2d(
        in_channels=self.channels[3],
        out_channels=self.channels[2],
        kernel_size=self.up_conv_conf["kernel_size"],
        stride=self.up_conv_conf["stride"],
        padding=self.up_conv_conf["padding"]
    )

    self.block_3 = ModUNetBlock(
        in_channels=self.channels[3],
        out_channels=self.channels[2]
    )
    self.up_conv_3 = nn.ConvTranspose2d(
        in_channels=self.channels[2],
        out_channels=self.channels[1],
        kernel_size=self.up_conv_conf["kernel_size"],
        stride=self.up_conv_conf["stride"],
        padding=self.up_conv_conf["padding"]
    )

    self.block_4 = ModUNetBlock(
        in_channels=self.channels[2],
        out_channels=self.channels[1]
    )


  def __crop_and_join(self, prev_feature_map, x):
    cropped_prev_feature_map = transforms.functional.crop(prev_feature_map, top=0, left=0, height=x.shape[2], width=x.shape[3])
    return torch.cat((cropped_prev_feature_map, x), dim=1)

  def __resize_and_join(self, prev_feature_map, x):
    fixed_x = transforms.Resize(size=(prev_feature_map.shape[2], prev_feature_map.shape[3]), antialias=False)(x)
    return torch.cat((prev_feature_map, fixed_x), dim=1)

  def forward(self, x, prev_feature_maps):
    out_block_1 = self.block_1(self.__resize_and_join(prev_feature_maps[0], x))

    out_up_conv_1 = self.up_conv_1(out_block_1)

    out_block_2 = None

    if self.encoder_name == "segformer":
      if self.channel_mismatch_strategy == "crop":
        out_block_2 = self.block_2(self.__resize_and_join(prev_feature_maps[1][:, :256, :, :], out_up_conv_1))
      elif self.channel_mismatch_strategy == "upconv":
        out_block_2 = self.block_2(self.__resize_and_join(self.up_conv_320_256_channel(prev_feature_maps[1]), out_up_conv_1))
    else:
      out_block_2 = self.block_2(self.__resize_and_join(prev_feature_maps[1], out_up_conv_1))

    out_up_conv_2 = self.up_conv_2(out_block_2)

    out_block_3 = self.block_3(self.__resize_and_join(prev_feature_maps[2], out_up_conv_2))

    out_up_conv_3 = self.up_conv_3(out_block_3)

    out_block_4 = self.block_4(self.__resize_and_join(prev_feature_maps[3], out_up_conv_3))

    return out_block_4

## ModUnetTail

In [ ]:
class ModUNetTail(nn.Module):
  def __init__(self, num_classes, encoder_name, channels: List[int]):
    super().__init__()

    assert len(channels) == 6, "c:ModUNetTail p:channels needs to be of length 6"

    self.encoder_name = encoder_name
    self.channels = channels

    self.segmentation_tail = nn.Conv2d(
        in_channels=self.channels[1],
        out_channels=num_classes,
        kernel_size=1
    )
    if (
        encoder_name == "resnet18"
        or encoder_name == "resnet34"
        or encoder_name == "resnet50"
        or encoder_name == "resnet101"
        or encoder_name == "resnet152"
        or encoder_name == "segformer"
    ):
      self.upsize = nn.UpsamplingBilinear2d(scale_factor=4)

    self.activation_fn = nn.Sigmoid()

  def forward(self, x):
    if (
        self.encoder_name == "resnet18"
        or self.encoder_name == "resnet34"
        or self.encoder_name == "resnet50"
        or self.encoder_name == "resnet101"
        or self.encoder_name == "resnet152"
        or self.encoder_name == "segformer"):
      return self.activation_fn(self.upsize(self.segmentation_tail(x)))
    return self.activation_fn(self.segmentation_tail(x))


## ModUNet

In [ ]:
class ModUNet(nn.Module):
  def __init__(
      self,
      num_classes,
      encoder_name,
      pretrained:bool=False,
      channel_mismatch_strategy:str=None,
      channels: List[int]=[3, 64, 128, 256, 512, 1024],
      max_pool_conf: Dict={"kernel_size": 3, "stride": 2, "padding": 1},
      up_conv_conf: Dict={"kernel_size": 3, "stride": 2, "padding": 1},
      dropout_rate: float=0.2
    ):
    super().__init__()

    regular_channels = [3, 64, 128, 256, 512, 1024]
    deep_resnet_channels = [3, 256, 512, 1024, 2048, 4096]

    self.pretrained = pretrained
    self.channels = channels
    self.max_pool_conf = max_pool_conf
    self.up_conv_conf = up_conv_conf

    if encoder_name == "unet":
      self.enc = ModUNetEncoder(
          channels=self.channels,
          max_pool_conf=self.max_pool_conf
      )
    # The channels in the below ResNets are hard-coded (to allow for the use for pre-trained weights)
    elif encoder_name == "resnet18":
      self.channels = regular_channels
      self.enc = ResNet18Encoder(
          pretrained=self.pretrained
      )
    elif encoder_name == "resnet34":
      self.channels = regular_channels
      self.enc = ResNet34Encoder(
          pretrained=self.pretrained
      )
    elif encoder_name == "resnet50":
      self.channels = deep_resnet_channels
      self.enc = ResNet50Encoder(
          pretrained=self.pretrained
      )
    elif encoder_name == "resnet101":
      self.channels = deep_resnet_channels
      self.enc = ResNet101Encoder(
          pretrained=self.pretrained
      )
    elif encoder_name == "resnet152":
      self.channels = deep_resnet_channels
      self.enc = ResNet152Encoder(
          pretrained=self.pretrained
      )
    elif encoder_name == "modresnet18":
      self.channels = regular_channels
      self.enc = ModResNet18Encoder(
          pretrained=self.pretrained
      )
    elif encoder_name == "modresnet34":
      self.channels = regular_channels
      self.enc = ModResNet34Encoder(
          pretrained=self.pretrained
      )
    elif encoder_name == "modresnet50":
      self.channels = deep_resnet_channels
      self.enc = ModResNet50Encoder(
          pretrained=self.pretrained
      )
    elif encoder_name == "modresnet101":
      self.channels = deep_resnet_channels
      self.enc = ModResNet101Encoder(
          pretrained=self.pretrained
      )
    elif encoder_name == "modresnet152":
      self.channels = deep_resnet_channels
      self.enc = ModResNet152Encoder(
          pretrained=self.pretrained
      )
    elif (encoder_name == "segformer"):
      self.enc = SegformerEncoder.from_pretrained(
          "nvidia/mit-b5",
          output_hidden_states=True
      )
    else:
      raise AttributeError("c:ModUnet p:encoder_name was invalid")

    self.dropout = nn.Dropout2d(p=dropout_rate)

    self.center = ModUNetCenter(
        channels=self.channels,
        max_pool_conf=self.max_pool_conf,
        up_conv_conf=self.up_conv_conf
    )

    self.dec = ModUNetDecoder(
        encoder_name=encoder_name,
        channels=self.channels,
        up_conv_conf=self.up_conv_conf,
        channel_mismatch_strategy=channel_mismatch_strategy,
    )

    self.tail = ModUNetTail(
        num_classes=num_classes,
        encoder_name=encoder_name,
        channels=self.channels
    )

  def forward(self, x):
    out_enc = self.enc(x)
    out_center = self.center(self.dropout(out_enc[0]))
    out_dec = self.dec(out_center, out_enc)
    out_tail = self.tail(out_dec)

    return out_tail

In [ ]:
# x = torch.rand((1, 3, 352, 352))

# print(ModUNet(num_classes=2, encoder_name="unet")(x).shape)
# print(ModUNet(num_classes=2, encoder_name="resnet18")(x).shape)
# print(ModUNet(num_classes=2, encoder_name="resnet34")(x).shape)
# print(ModUNet(num_classes=2, encoder_name="resnet50")(x).shape)
# print(ModUNet(num_classes=2, encoder_name="resnet101")(x).shape)
# print(ModUNet(num_classes=2, encoder_name="resnet152")(x).shape)

In [ ]:
# print(ModUNet(num_classes=2, encoder_name="segformer", channel_mismatch_strategy="upconv")(x).shape)

In [ ]:
# x = torch.rand((1, 3, 352, 352))

In [ ]:
# bench_modunet_encoder = ModUNetEncoder(
#     channels=[3, 64, 128, 256, 512, 1024],
#     max_pool_conf={
#       "kernel_size": 3,
#       "stride": 2,
#       "padding": 1
#     }
# )

# out_modunet_encoder = bench_modunet_encoder(x)

# for ten in out_modunet_encoder:
#   print(ten.shape)



In [ ]:
# bench_resnet34 = ResNet34Encoder(True)

# out_bench_resnet34 = bench_resnet34(x)

# for ten in out_bench_resnet34:
#   print(ten.shape)

# self.channels = [3, 64, 128, 256, 512, 1024]
# self.channels = [3, 64, 64, 128, 256, 512, ]

In [ ]:
# bench_resnet50 = ResNet50Encoder(True)

# out_bench_resnet50 = bench_resnet50(x)

# for ten in out_bench_resnet50:
#   print(ten.shape)

# channels = [3, 256, 512, 1024, 2048, 4096]

In [ ]:
# bench_segformer_unet = ModUNet(
#     num_classes=12, # len(label_map)
#     encoder_name="segformer",
#     channel_mismatch_strategy="crop"
# )

In [ ]:
# bench_segformer_unet.to("cuda")

In [ ]:
# bench_unet_encoder = ModUNetEncoder()
# bench_unet_center = ModUNetCenter()
# bench_unet_decoder = ModUNetDecoder("unet")
# bench_unet_tail = ModUNetTail(2, "unet")

# out_enc = bench_unet_encoder(x)

# print(f"512 Channel Shape Feature Map: {out_enc[0].shape}")
# print(f"256 Channel Shape Feature Map: {out_enc[1].shape}")
# print(f"128 Channel Shape Feature Map: {out_enc[2].shape}")
# print(f"64 Channel Shape Feature Map: {out_enc[3].shape}")

# out_center = bench_unet_center(out_enc[0])

# print(f"\nout_center.shape {out_center.shape}")

# out_dec = bench_unet_decoder(out_center, out_enc)

# print(f"\nout_dec.shape {out_dec.shape}")

# out_tail = bench_unet_tail(out_dec)
# print(f"\nout_tail.shape {out_tail.shape}")

In [ ]:
# bench_resnet_encoder = ResNet34Encoder()

# out_enc_resnet = bench_resnet_encoder(x)

# print(f"512 Channel Shape Feature Map: {out_enc_resnet[0].shape}")
# print(f"256 Channel Shape Feature Map: {out_enc_resnet[1].shape}")
# print(f"128 Channel Shape Feature Map: {out_enc_resnet[2].shape}")
# print(f"64 Channel Shape Feature Map: {out_enc_resnet[3].shape}")

# out_center_resnet = bench_unet_center(out_enc_resnet[0])

# print(f"\nout_center_resnet.shape {out_center_resnet.shape}")


# out_dec_resnet = bench_unet_decoder(out_center_resnet, out_enc_resnet)

# print(f"\nout_dec.shape {out_dec_resnet.shape}")

# out_tail_resnet = bench_unet_tail(out_dec_resnet)

# print(f"\nout_tail_resnet.shape {out_tail_resnet.shape}")

In [ ]:
# bench_segformer_encoder = SegformerEncoder.from_pretrained(
#   "nvidia/mit-b5",
#   output_hidden_states=True
# )
# bench_unet_center_for_segformer_enc = ModUNetCenter()
# bench_unet_decoder_for_segformer_enc = ModUNetDecoder(
#     encoder_name="segformer",
#     channel_mismatch_strategy="upconv"
# )
# bench_unet_tail_for_segformer_enc = ModUNetTail(2, "unet")

# out_enc = bench_segformer_encoder(x)

# print(f"512 Channel Shape Feature Map: {out_enc[0].shape}")
# print(f"256 Channel Shape Feature Map: {out_enc[1].shape}")
# print(f"256 Channel Shape Feature Map after applied Conv: {conv2d(out_enc[1]).shape}")
# print(f"128 Channel Shape Feature Map: {out_enc[2].shape}")
# print(f"64 Channel Shape Feature Map: {out_enc[3].shape}")

# out_center = bench_unet_center_for_segformer_enc(out_enc[0])

# print(f"\nout_center.shape {out_center.shape}")

# out_dec = bench_unet_decoder_for_segformer_enc(out_center, out_enc)

# print(f"\nout_dec.shape {out_dec.shape}")

# out_tail = bench_unet_tail_for_segformer_enc(out_dec)

# print(f"\nout_tail.shape {out_tail.shape}")

In [ ]:
# bench_unet = ModUNet(num_classes=2, encoder_name="resnet34")

In [ ]:
# bench_unet(x).shape

UNet preliminary testing

Define the image and mask

In [ ]:
# img_transform = transforms.Compose(
#     [
#         transforms.Resize(size=(352, 352), antialias=True),
#         lambda x: x.to(torch.float) / 255,
#         transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
#     ]
# )

# mask_transform = transforms.Compose(
#     [
#         transforms.Resize(size=(352, 352), antialias=True),
#     ]
# )

Define the training and validation dataset

In [ ]:
# label_map = {
#         "sky": 0,
#         "building": 1,
#         "pole": 2,
#         "road": 3,
#         "pavement": 4,
#         "tree": 5,
#         "signsymbol": 6,
#         "fence": 7,
#         "car": 8,
#         "pedestrain": 9,
#         "bicyclist": 10,
#         "unlabelled": 11
# }

# train_data = SegmentationDatasetSeperateMasks(
#     img_dir="drive/MyDrive/CamVid/train",
#     mask_dir="drive/MyDrive/CamVid/trainannot",
#     label_map=label_map,
#     img_transform=img_transform,
#     mask_transform=mask_transform,
# )

# valid_data = SegmentationDatasetSeperateMasks(
#     img_dir="drive/MyDrive/CamVid/val",
#     mask_dir="drive/MyDrive/CamVid/valannot",
#     label_map=label_map,
#     img_transform=img_transform,
#     mask_transform=mask_transform,
#     mask_type=torch.float,
# )

Define the training and validation dataloader

In [ ]:
# train_dataloader = DataLoader(train_data, batch_size=8)
# valid_dataloader = DataLoader(valid_data, batch_size=8)

Define the model

In [ ]:
# model = ModUNet(
#   num_classes=len(label_map),
#   encoder_name="resnet34"
# ).to(DEVICE)

Define the loss function

In [ ]:
# loss_fn = DiceLoss()

Define the optimizer

In [ ]:
# optimizer = torch.optim.Adam(
#     bench_segformer_unet.parameters(),
#     lr=1e-4
# )

In [ ]:
# def train_loop(dataloader, model, loss_fn, optimizer, device):
#   size = len(dataloader.dataset)

#   model.train()

#   for batch, (X, y) in enumerate(dataloader):
#     X = X.to(device)
#     y = y.to(device)

#     # Compute the prediction and loss
#     pred = model(X)
#     loss = loss_fn(pred, y)

#     # Perform backprop
#     loss.backward()
#     optimizer.step()
#     optimizer.zero_grad()

#     loss, current = loss.item(), (batch + 1) * len(X)
#     print(f"loss: {loss:7f} [{current:>5d}/{size:5d}]")

In [ ]:
# %run "/content/drive/MyDrive/Colab Notebooks/epochs.ipynb"
# train_epoch = TrainEpoch(
#     dataloader=train_dataloader,
#     model=model,
#     optimizer=optimizer,
#     loss_fn=loss_fn,
#     metrics=[],
#     device=DEVICE
# )

In [ ]:
# for i in range(10):
#   print(f"Epoch {i + 1}: ")
#   train_loop(train_dataloader, bench_segformer_unet, loss_fn, optimizer, DEVICE)

In [ ]:
# for i in range(10):
#   print(f"Epoch {i + 1}: ")
#   train_loop(train_dataloader, bench_segformer_unet, loss_fn, optimizer, DEVICE)

In [ ]:
# # # Segformer run
# for i in range(10):
#   print(f"Epoch {i + 1}: ")
#   train_loop(train_dataloader, bench_segformer_unet, loss_fn, optimizer, DEVICE)

In [ ]:
# torch.save(model.state_dict(), "/content/drive/MyDrive/checkpoints/segformer_mod_unet.pth")


In [ ]:
# !ls "/content/drive/MyDrive/checkpoints/"

In [ ]:
# # ResNet run
# for i in range(20):
#   print(f"Epoch {i + 1}: ")
#   train_epoch.run()

In [ ]:
# optimizer.param_groups[0]['lr'] = 1e-5

In [ ]:
# !pip3 install segmentation_models_pytorch

In [ ]:

# import segmentation_models_pytorch as smp

# import segmentation_models_pytorch.utils.metrics


# metrics = [
#     smp.utils.metrics.IoU(threshold=0.5),
# ]

# model.eval()

# loss_fn.__name__ = "lol"

# test_epoch = smp.utils.train.ValidEpoch(
#     model=model,
#     loss=loss_fn,
#     metrics=metrics,
#     device="cuda",
# )

# logs = test_epoch.run(valid_dataloader)

In [ ]:
# x = train_data[0][0].unsqueeze(0)

In [ ]:
# with torch.no_grad():
#   pred_y = model.forward(x)

In [ ]:
# ground_y = train_data[0][1].unsqueeze(0)

In [ ]:
# plt.imshow(x.squeeze().cpu().permute(1, 2, 0) )

In [ ]:
# plt.imshow(pred_y.squeeze()[0].unsqueeze(0).cpu().permute(1, 2, 0).round(), cmap="gray")

In [ ]:
# plt.imshow(pred_y.squeeze()[1].unsqueeze(0).cpu().permute(1, 2, 0).round(), cmap="gray")

Analyzing model outputs

In [ ]:
# encs = {
#     "resnet18": ModUNet(num_classes=7,encoder_name="resnet18",pretrained=True),
#     "modresnet18": ModUNet(num_classes=7,encoder_name="modresnet18",pretrained=True),
#     # "resnet34": ResNet34Encoder(),
#     # "resnet50": ResNet50Encoder(),
#     # "resnet101": ResNet101Encoder(),
#     # "resnet152": ResNet152Encoder()
# }

# x = torch.randn((1, 3, 256, 256))
# for name_enc, enc in encs.items():
#   print(name_enc)
#   enc_out = enc(x)
#   for out in enc_out:
#     print(out.shape)
#   print()

In [ ]:
# class ModResNet18Encoder(nn.Module):
#   def __init__(self, pretrained=False):
#     super().__init__()

#     self.conv1 = ModUNetBlock(
#         in_channels=3,
#         out_channels=64,
#     )

#     self.conv2_x = nn.Sequential(OrderedDict([
#         ("layer1", ResNetLayer(in_channels=64, out_channels=64, num_blocks=2))
#     ]))

#     self.conv3_x = ResNetLayer(
#         in_channels=64,
#         out_channels=128,
#         num_blocks=2
#     )

#     self.conv4_x = ResNetLayer(
#         in_channels=128,
#         out_channels=256,
#         num_blocks=2
#     )

#     self.conv5_x = ResNetLayer(
#         in_channels=256,
#         out_channels=512,
#         num_blocks=2
#     )

#     if pretrained:
#       self.load_pretrained_weights()

#   def load_pretrained_weights(self):
#     # Declare a pretrained PyTorch classification ResNet
#     pretrained_model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', weights="ResNet18_Weights.DEFAULT")
#     pretrained_state_dict = pretrained_model.state_dict()

#     # Deleted the unwanted weights
#     del pretrained_state_dict["fc.weight"]
#     del pretrained_state_dict["fc.bias"]

#     # Creating a state dict with the correct layer names as they are listed internally
#     correct_layer_names_pretrained_state_dict = OrderedDict()

#     # Rename the pretrained_state_dict layers as they correspond internally
#     for new_key, old_key in zip(self.state_dict().keys(), pretrained_state_dict.keys()):
#       correct_layer_names_pretrained_state_dict[new_key] = pretrained_state_dict[old_key]

#     # Load the new_state_dict
#     self.load_state_dict(correct_layer_names_pretrained_state_dict)

#   def forward(self, x):
#     # The below names follow the listed in the paper
#     conv1_out = self.conv1(x)

#     conv2_x_out = self.conv2_x(conv1_out)

#     conv3_x_out = self.conv3_x(conv2_x_out)

#     conv4_x_out = self.conv4_x(conv3_x_out)

#     conv5_x_out = self.conv5_x(conv4_x_out)

#     return conv5_x_out, conv4_x_out, conv3_x_out, conv2_x_out, conv1_out

# encs = {
#     "modunet": ModUNetEncoder([3, 64, 128, 256, 512, 1024], {"kernel_size": 3, "stride": 2, "padding": 1}),
#     "modresnet18": ModResNet18Encoder(),
# }

# x = torch.randn((1, 3, 256, 256))
# for name_enc, enc in encs.items():
#   print(name_enc)
#   enc_out = enc(x)
#   for out in enc_out:
#     print(out.shape)
  # print()